In [2]:
import os
from uuid import UUID
import uuid
import time
import json
from pathlib import Path
import tiktoken
from collections import deque

from qdrant import VectorDB
from qdrant_client import models
from utils import load_config, get_type
from dotenv import load_dotenv
from batches import *
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output
from google import genai
from google.genai import types
from openai import OpenAI

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")

In [5]:
config = load_config("config.ini")

# Connect to QDRANT
qdrant_client = VectorDB(
    port=config.getint("QDRANT", "port"),
    host=config.get("QDRANT", "host")
)

[2026-02-26 15:23:42] Connected to http://localhost:6333/dashboard
[2026-02-26 15:23:42] Storage will be at /Users/pablobedolla/DlrowSreckah/tuutrag-open/tuutrag/data/qdrant


In [5]:
# Connect to OpenAI
openai_client = OpenAI(api_key=openai_key)

# Connect to Gemini
gemini_client = genai.Client(api_key=gemini_key)

In [53]:
# Create branch chunk collection
qdrant_client.create_collection(
    collection_name="branch_chunks",
    vector_params=models.VectorParams(
        size=3072,
        distance=models.Distance.COSINE
    )
)

In [54]:
GEMINI_MODEL = "gemini-embedding-001"
TASK_TYPE = "SEMANTIC_SIMILARITY"

In [55]:
with open("../data/fixed_size_chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

In [22]:
SCHEME = 'o200k_base'
enc = tiktoken.get_encoding(SCHEME)
total_tokens = 0

for idx, obj in enumerate(all_chunks):
    total_tokens += len(enc.encode(obj['chunk']))

print(f"{total_tokens:,}")

18,907,969


In [61]:
# Responses <-- gemini API
responses = []

In [81]:
RPM = 3_000
TPM = 800_000
BATCH_LIMIT = 100
MIN_BATCH_INTERVAL = (BATCH_LIMIT / RPM) * 60

In [82]:
# Batches <-- List[lists]
batches = []

for idx in range(0, len(all_chunks), BATCH_LIMIT):
    temp = all_chunks[idx:idx + BATCH_LIMIT]
    batches.append(temp)

In [83]:
window_start = time.monotonic()
window_tokens = 0
window_requests = 0

for idx, batch in enumerate(batches):
    batch_start = time.monotonic()
    batch_tokens = sum(len(enc.encode(chunk['chunk'])) for chunk in batch)

    over_rpm = (window_requests + len(batch)) > RPM
    over_tpm = (window_tokens + batch_tokens) > TPM
    window_elapsed = batch_start - window_start

    if window_elapsed < 60 and (over_rpm or over_tpm):
        clear_output(wait=True)
        print(f"=== LIMIT HIT – waiting 60s before batch {idx + 1}/{len(batches)} "
              f"[RPM: {window_requests}/{RPM} | TPM: {window_tokens}/{TPM}] ===")
        time.sleep(80)
        window_start = time.monotonic()
        window_tokens = 0
        window_requests = 0

    window_tokens += batch_tokens
    window_requests += len(batch)

    clear_output(wait=True)
    print(f"Processing batch {idx + 1}/{len(batches)} | "
          f"RPM: {window_requests}/{RPM} | TPM: {window_tokens}/{TPM}")

    result = gemini_client.models.embed_content(
        model=GEMINI_MODEL,
        contents=[chunk['chunk'] for chunk in batch],
        config=types.EmbedContentConfig(
            task_type="SEMANTIC_SIMILARITY",
            output_dimensionality=3072
        )
    )

    if len(result.embeddings) != len(batch):
        raise ValueError(f"Batch {idx}: expected {len(batch)} embeddings, got {len(result.embeddings)}")

    for jdx, embedding in enumerate(result.embeddings):
        responses.append({
            **batch[jdx],
            "embedding": embedding.values,
            "qdrant-id": uuid.uuid5(uuid.NAMESPACE_DNS, str(batch[jdx]["uuid"]))
        })

    elapsed = time.monotonic() - batch_start
    sleep_time = MIN_BATCH_INTERVAL - elapsed
    if sleep_time > 0:
        time.sleep(sleep_time)

Processing batch 245/245 | RPM: 178/3000 | TPM: 80395/800000


In [87]:
# Fetch all existing IDs in the collection
existing_ids = set()
offset = None
while True:
    result, offset = qdrant_client.client.scroll(
        collection_name="branch_chunks",
        offset=offset,
        limit=1000,
        with_payload=False,
        with_vectors=False
    )
    existing_ids.update(str(p.id) for p in result)
    if offset is None:
        break

print(f"Found {len(existing_ids)} existing points, skipping...")

pending = [obj for obj in responses if str(obj["qdrant-id"]) not in existing_ids]
print(f"Upserting {len(pending)} new points...")

UPSERT_BATCH_SIZE = 100

for i in range(0, len(pending), UPSERT_BATCH_SIZE):
    batch = pending[i:i + UPSERT_BATCH_SIZE]
    qdrant_client.client.upsert(
        collection_name="branch_chunks",
        wait=True,
        points=[
            models.PointStruct(
                id=str(obj["qdrant-id"]),
                vector=obj["embedding"],
                payload={
                    "uuid": str(obj["uuid"]),
                    "uuid_prefix": str(obj["uuid"]).split(".")[0],
                    "type": get_type(str(obj["uuid"]).split(".")[0]),
                    "chunk": obj["chunk"],
                    "path": obj.get("path", ""),  # safe fallback for missing key
                }
            )
            for obj in batch
        ]
    )
    if (i // UPSERT_BATCH_SIZE) % 10 == 0:
        print(f"Upserted {min(i + UPSERT_BATCH_SIZE, len(pending))}/{len(pending)}")

Found 26000 existing points, skipping...
Upserting 11278 new points...
Upserted 100/11278
Upserted 1100/11278
Upserted 2100/11278
Upserted 3100/11278
Upserted 4100/11278
Upserted 5100/11278
Upserted 6100/11278
Upserted 7100/11278
Upserted 8100/11278
Upserted 9100/11278
Upserted 10100/11278
Upserted 11100/11278


In [79]:
processed_uuids = {obj['uuid'] for obj in responses}
all_chunks = [chunk for chunk in all_chunks if chunk['uuid'] not in processed_uuids]

In [7]:
with open("../data/responses.json", "r", encoding="utf-8") as f:
    responses = json.load(f)